In [1]:
from pathlib import Path

import anndata as ad
import napari
import numpy as np
import pandas as pd


def _image_table(adata):
    """Image-level measurements (FileName_*, PathName_*, Metadata_*, ...) as a DataFrame indexed
    by ImageNumber -- the AnnData equivalent of the per-image columns a CSV export joins onto
    every object row."""
    return pd.DataFrame(adata.uns["cellprofiler"]["image"]).set_index("ImageNumber")


def _cell_locations(cells):
    """(image_path, points in napari's row/col == Y/X order) for a table of cells already
    restricted to a single image -- either a CSV export (DataFrame) or an AnnData export."""
    if isinstance(cells, ad.AnnData):
        image_numbers = cells.obs["ImageNumber"].unique()
        assert len(image_numbers) == 1
        image_row = _image_table(cells).loc[int(image_numbers[0])]
        points = cells.obsm["spatial"][:, [1, 0]]  # obsm["spatial"] is (X, Y); napari wants (Y, X)
    else:
        assert len(cells["FileName_DNA"].unique()) == 1
        image_row = cells.iloc[0]
        points = cells[["Location_Center_Y", "Location_Center_X"]].to_numpy()
    image_path = Path(image_row["PathName_DNA"], image_row["FileName_DNA"])
    return image_path, points


def _nucleus_locations(cells, nuc_table=None):
    """Nucleus center points (napari's Y, X order), one row per row of `cells`, in the same order.

    For an AnnData export the nucleus is already joined onto every cell row, as the obs columns
    `Nuclei__Center_X/Y` -- not `.X`: ExportToAnnData keeps Location/orientation measurements out
    of X/var (they'd bias morphological similarity on where an object sits in the image, not its
    biology) and reports them in obs instead. For a CSV export the two tables are separate and must
    be joined by hand via `Parent_Nuclei` -> the nucleus table's `ObjectNumber`, within the same
    `ImageNumber` -- pass the Nuclei table (e.g. `nuc_table`) as `nuc_table`.
    """
    if isinstance(cells, ad.AnnData):
        return cells.obs[["Nuclei__Center_Y", "Nuclei__Center_X"]].to_numpy()
    if nuc_table is None:
        raise ValueError("nuc_table is required to look up nucleus centers for a CSV export.")
    nuc_by_key = nuc_table.reset_index().set_index(["ImageNumber", "ObjectNumber"])
    keys = list(zip(cells.index, cells["Parent_Nuclei"]))
    return nuc_by_key.loc[keys, ["Location_Center_Y", "Location_Center_X"]].to_numpy()


def show_cells(cell_table, pattern):
    """Open the matching cells' image and display their centers in Napari.

    `cell_table` is either an ExportToSpreadsheet CSV (a DataFrame) or an ExportToAnnData export
    (an AnnData); `pattern` is the matching boolean mask/array for that table. One single image
    is expected for the given pattern.
    """
    cells = cell_table[pattern] if isinstance(cell_table, ad.AnnData) else cell_table.loc[pattern]
    image_path, points = _cell_locations(cells)
    assert image_path.exists(), f"Image file {image_path} does not exist."

    viewer = napari.Viewer()
    viewer.open(image_path)
    viewer.add_points(
        points,
        size=8,
        face_color="green",
        name="Cells",
    )
    return viewer


def show_cells_for_image(cell_table, image_filename, filename_column="FileName_DNA"):
    """Display all cells belonging to one image."""
    if isinstance(cell_table, ad.AnnData):
        image_numbers = _image_table(cell_table)
        matching = image_numbers.index[image_numbers[filename_column].eq(image_filename)]
        pattern = cell_table.obs["ImageNumber"].isin(matching).to_numpy()
    else:
        pattern = cell_table[filename_column].eq(image_filename)
    return show_cells(cell_table, pattern)


def show_cells_and_nuclei(cell_table, pattern, nuc_table=None):
    """Like `show_cells`, but also overlays each cell's matching nucleus center in red.

    `nuc_table` (the Nuclei CSV export) is required when `cell_table` is a DataFrame; it is
    ignored for an AnnData export, which already carries the joined nucleus centers.
    """
    cells = cell_table[pattern] if isinstance(cell_table, ad.AnnData) else cell_table.loc[pattern]
    image_path, cell_points = _cell_locations(cells)
    nucleus_points = _nucleus_locations(cells, nuc_table)
    assert image_path.exists(), f"Image file {image_path} does not exist."

    viewer = napari.Viewer()
    viewer.open(image_path)
    viewer.add_points(cell_points, size=8, face_color="green", name="Cells")
    viewer.add_points(nucleus_points, size=5, face_color="red", name="Nuclei")
    return viewer


def show_cells_and_nuclei_for_image(cell_table, image_filename, nuc_table=None, filename_column="FileName_DNA"):
    """Display all cells and their nuclei belonging to one image."""
    if isinstance(cell_table, ad.AnnData):
        image_numbers = _image_table(cell_table)
        matching = image_numbers.index[image_numbers[filename_column].eq(image_filename)]
        pattern = cell_table.obs["ImageNumber"].isin(matching).to_numpy()
    else:
        pattern = cell_table[filename_column].eq(image_filename)
    return show_cells_and_nuclei(cell_table, pattern, nuc_table=nuc_table)

Input:
* list of object table files ("<prefix><object-name>.csv")
* Reference object name for aggregation (default: "Nuclei")
* Aggregation function for multiple child objects (default: mean)

Process: 
* Read each table
* Build lineage DAG tree between objects from reference (root node):
    * Check for all other objects (if any) if they have "Parent_<reference>" in their features
        * If so, they're child nodes from the root node
    * Iterate by looking for "Parent_<rooted>" for any connected object type until all object types are connected or no new links can be established
    * Drop any disconnected components with a warning
* Concatenate any child object type or related object type to the reference objects (wide format), either as-is (for 1-to-1 mappings) or using the chosen aggregation function (for 1-to-many mappings)
* Create AnnData object
    * Measurements to `.X`
    * Metadata to `.obs`
     

Format:
* Reusable import functions (to be packaged and distributed later with other related tasks)

## From spreadsheet export

In [2]:
cell_table = pd.read_csv("Testrun_cp2sd_Cells.csv", index_col=0)
cell_table.describe()

,ObjectNumber,Metadata_Channel,Metadata_Column,Metadata_Field,Metadata_FileLocation,Metadata_Frame,Metadata_Row,Metadata_Series,AreaShape_Area,AreaShape_BoundingBoxArea,...,Location_MaxIntensity_X_LogProtein,Location_MaxIntensity_X_LogTubulin,Location_MaxIntensity_Y_LogDNA,Location_MaxIntensity_Y_LogProtein,Location_MaxIntensity_Y_LogTubulin,Location_MaxIntensity_Z_LogDNA,Location_MaxIntensity_Z_LogProtein,Location_MaxIntensity_Z_LogTubulin,Number_Object_Number,Parent_Nuclei
count,151.000000,0.0,151.000000,151.000000,0.0,151.0,151.000000,151.0,151.000000,151.000000,...,151.000000,151.000000,151.000000,151.000000,151.000000,151.0,151.0,151.0,151.000000,151.000000
mean,39.019868,NaN,5.271523,3.238411,NaN,0.0,5.569536,0.0,6692.238411,11412.430464,...,589.291391,582.470199,536.377483,538.152318,535.523179,0.0,0.0,0.0,39.019868,39.019868
std,30.715788,NaN,1.025890,1.081410,NaN,0.0,2.099238,0.0,3237.981313,6054.165977,...,355.287256,356.662470,306.263802,306.355431,309.452222,0.0,0.0,0.0,30.715788,30.715788
min,1.000000,NaN,2.000000,1.000000,NaN,0.0,1.000000,0.0,1181.000000,2064.000000,...,35.000000,14.000000,19.000000,3.000000,0.000000,0.0,0.0,0.0,1.000000,1.000000
25%,12.000000,NaN,5.000000,3.000000,NaN,0.0,3.000000,0.0,4751.500000,7393.500000,...,296.500000,292.000000,276.500000,281.000000,277.500000,0.0,0.0,0.0,12.000000,12.000000
50%,30.000000,NaN,5.000000,3.000000,NaN,0.0,7.000000,0.0,6110.000000,10600.000000,...,515.000000,522.000000,531.000000,553.000000,540.000000,0.0,0.0,0.0,30.000000,30.000000
75%,64.500000,NaN,5.000000,3.000000,NaN,0.0,7.000000,0.0,8312.500000,13747.500000,...,857.500000,862.500000,838.000000,852.000000,833.000000,0.0,0.0,0.0,64.500000,64.500000
max,102.000000,NaN,7.000000,5.000000,NaN,0.0,7.000000,0.0,21607.000000,39590.000000,...,1333.000000,1317.000000,1023.000000,1020.000000,1022.000000,0.0,0.0,0.0,102.000000,102.000000


In [3]:
nuc_table = pd.read_csv("Testrun_cp2sd_Nuclei.csv", index_col=0)
nuc_table.describe()

,ObjectNumber,Metadata_Channel,Metadata_Column,Metadata_Field,Metadata_FileLocation,Metadata_Frame,Metadata_Row,Metadata_Series,AreaShape_Area,AreaShape_BoundingBoxArea,...,Location_MaxIntensity_X_LogDNA,Location_MaxIntensity_X_LogProtein,Location_MaxIntensity_X_LogTubulin,Location_MaxIntensity_Y_LogDNA,Location_MaxIntensity_Y_LogProtein,Location_MaxIntensity_Y_LogTubulin,Location_MaxIntensity_Z_LogDNA,Location_MaxIntensity_Z_LogProtein,Location_MaxIntensity_Z_LogTubulin,Number_Object_Number
count,151.000000,0.0,151.000000,151.000000,0.0,151.0,151.000000,151.0,151.000000,151.000000,...,151.000000,151.000000,151.000000,151.000000,151.000000,151.000000,151.0,151.0,151.0,151.000000
mean,39.019868,NaN,5.271523,3.238411,NaN,0.0,5.569536,0.0,1642.139073,2450.834437,...,587.317881,587.569536,586.907285,536.556291,536.596026,536.337748,0.0,0.0,0.0,39.019868
std,30.715788,NaN,1.025890,1.081410,NaN,0.0,2.099238,0.0,774.669866,1373.300455,...,357.594377,357.461653,355.963057,306.500411,306.268818,310.301292,0.0,0.0,0.0,30.715788
min,1.000000,NaN,2.000000,1.000000,NaN,0.0,1.000000,0.0,729.000000,891.000000,...,19.000000,35.000000,33.000000,19.000000,8.000000,22.000000,0.0,0.0,0.0,1.000000
25%,12.000000,NaN,5.000000,3.000000,NaN,0.0,3.000000,0.0,1191.000000,1620.000000,...,293.500000,284.500000,294.500000,275.000000,281.000000,269.500000,0.0,0.0,0.0,12.000000
50%,30.000000,NaN,5.000000,3.000000,NaN,0.0,7.000000,0.0,1429.000000,2050.000000,...,493.000000,505.000000,494.000000,531.000000,539.000000,523.000000,0.0,0.0,0.0,30.000000
75%,64.500000,NaN,5.000000,3.000000,NaN,0.0,7.000000,0.0,1940.500000,2906.000000,...,855.000000,856.500000,860.000000,838.000000,824.000000,837.500000,0.0,0.0,0.0,64.500000
max,102.000000,NaN,7.000000,5.000000,NaN,0.0,7.000000,0.0,6829.000000,11868.000000,...,1313.000000,1333.000000,1339.000000,992.000000,1005.000000,1014.000000,0.0,0.0,0.0,102.000000


In [4]:
viewer = show_cells_for_image(cell_table, "r02c04f01ch1.tiff")

In [5]:
viewer = show_cells_and_nuclei_for_image(cell_table, "r02c04f01ch1.tiff", nuc_table=nuc_table)

## From anndata export

In [6]:
# Matched content (see cp_export/tests/test_fidelity.py)
adata = ad.read_h5ad("Testrun_cp2sd_2ad.h5ad")

In [7]:
viewer = show_cells_for_image(adata, "r02c04f01ch1.tiff")

In [8]:
viewer = show_cells_and_nuclei_for_image(adata, "r02c04f01ch1.tiff")

In [9]:
adata.uns['cellprofiler_mapping']

{'channels':       channel module_name  module_num    source
 0      LogDNA   ImageMath           5  pipeline
 1  LogProtein   ImageMath           6  pipeline
 2  LogTubulin   ImageMath           7  pipeline,
 'measurements':      object                    cp_name               module_name  module_num  \
 0    Nuclei          Location_Center_X    IdentifyPrimaryObjects           8   
 1    Nuclei          Location_Center_Y    IdentifyPrimaryObjects           8   
 2    Nuclei          Location_Center_Z    IdentifyPrimaryObjects           8   
 3    Nuclei       Number_Object_Number    IdentifyPrimaryObjects           8   
 4    Nuclei       Children_Cells_Count  IdentifySecondaryObjects           9   
 ..      ...                        ...                       ...         ...   
 181   Cells       AreaShape_FormFactor    MeasureObjectSizeShape          11   
 182   Cells         AreaShape_Solidity    MeasureObjectSizeShape          11   
 183   Cells       AreaShape_ConvexArea    Mea

In [10]:
adata.uns['cellprofiler_mapping']['measurements']

,object,cp_name,module_name,module_num,category,channel,channel2,destination,anndata_name,reason
0,Nuclei,Location_Center_X,IdentifyPrimaryObjects,8,Location,,,obs,Nuclei__Center_X,"position, orientation, or identity/linkage, no..."
1,Nuclei,Location_Center_Y,IdentifyPrimaryObjects,8,Location,,,obs,Nuclei__Center_Y,"position, orientation, or identity/linkage, no..."
2,Nuclei,Location_Center_Z,IdentifyPrimaryObjects,8,Location,,,obs,Nuclei__Location_Center_Z,"position, orientation, or identity/linkage, no..."
3,Nuclei,Number_Object_Number,IdentifyPrimaryObjects,8,Number,,,obs,Nuclei__Number_Object_Number,"position, orientation, or identity/linkage, no..."
4,Nuclei,Children_Cells_Count,IdentifySecondaryObjects,9,Children,,,obs,Nuclei__Children_Cells_Count,"position, orientation, or identity/linkage, no..."
...,...,...,...,...,...,...,...,...,...,...
181,Cells,AreaShape_FormFactor,MeasureObjectSizeShape,11,AreaShape,,,X,Cells__FormFactor,
182,Cells,AreaShape_Solidity,MeasureObjectSizeShape,11,AreaShape,,,X,Cells__Solidity,
183,Cells,AreaShape_ConvexArea,MeasureObjectSizeShape,11,AreaShape,,,X,Cells__ConvexArea,
184,Cells,AreaShape_Compactness,MeasureObjectSizeShape,11,AreaShape,,,X,Cells__Compactness,
